# Developer Reference Datasets — a quick tour

Computed lookup tables web & app developers reach for: **aspect ratios, WCAG 2.1 contrast, MIME + magic numbers, UTF-8/16/32 sizes, QR capacity, and browser format support.** Every value is computed from the spec (not scraped), CC BY 4.0.

Docs & code: https://github.com/cleanor-app/developer-reference-datasets · DOI [10.5281/zenodo.21369536](https://doi.org/10.5281/zenodo.21369536)


In [ ]:
import pandas as pd, glob, os
import matplotlib.pyplot as plt
plt.rcParams['figure.dpi'] = 120

def load(name):
    for p in [f'/kaggle/input/developer-reference-datasets/{name}', f'data/{name}', name]:
        if os.path.exists(p): return pd.read_csv(p)
    raise FileNotFoundError(name)

print('CSV files:', sorted(os.path.basename(p) for p in glob.glob('/kaggle/input/**/*.csv', recursive=True)) or 'run locally')


## 1. WCAG contrast — which greys pass on white?
AA needs a contrast ratio of 4.5:1 for normal text. The line on white sits around gray-500.


In [ ]:
wb = load('wcag-on-white-black.csv')
on_white = wb[wb['vs']=='white'].sort_values('contrast_ratio')
print(on_white[['color_name','hex','contrast_ratio','aa_normal','aaa_normal']].to_string(index=False))
ax = on_white.plot.barh(x='color_name', y='contrast_ratio', legend=False, color='#4576fd', figsize=(7,6))
ax.axvline(4.5, color='#e0362f', ls='--', label='AA 4.5:1'); ax.axvline(7, color='#0f9e72', ls='--', label='AAA 7:1')
ax.set_xlabel('contrast ratio vs white'); ax.set_title('WCAG contrast on white'); ax.legend(); plt.tight_layout()


## 2. Character encoding — bytes per character by script
A Latin char is 1 UTF-8 byte, Greek/Cyrillic 2, CJK 3, most emoji 4.


In [ ]:
enc = load('encoding-sizes.csv').sort_values('utf8_bytes_per_code_point')
print(enc[['script','code_points','utf8_bytes','utf8_bytes_per_code_point']].to_string(index=False))
ax = enc.plot.barh(x='script', y='utf8_bytes_per_code_point', legend=False, color='#7d5cff', figsize=(7,5))
ax.set_xlabel('UTF-8 bytes per code point'); ax.set_title('How many bytes is a character?'); plt.tight_layout()


## 3. QR code capacity — how much fits?
A version-1 QR holds 41 digits / 25 alphanumeric / 17 bytes at ECC L.


In [ ]:
qr = load('qr-code-capacity.csv')
print(qr[qr['version']==1][['ecc_level','numeric','alphanumeric','byte','kanji']].to_string(index=False))
L = qr[qr['ecc_level']=='L']
ax = L.plot(x='version', y=['numeric','alphanumeric','byte'], figsize=(7,4))
ax.set_ylabel('max characters (ECC L)'); ax.set_title('QR capacity by version'); plt.tight_layout()


## 4. Aspect ratios & format support



In [ ]:
ar = load('aspect-ratios.csv')
print('16:9 at common widths:')
print(ar[ar['ratio']=='16:9'][['width_px','height_px']].to_string(index=False))
fs = load('image-format-support.csv')
print('\nFirst full support:')
print(fs[fs['browser'].isin(['chrome','firefox','safari'])].pivot(index='format', columns='browser', values='first_full_support_version').to_string())


---
All tables are reproducible — the generators live in the [GitHub repo](https://github.com/cleanor-app/developer-reference-datasets). Reuse under CC BY 4.0 with attribution to Cleanor Labs.
